In [1]:
# ============================================================
# K-MEANS + NLP CLUSTERING FOR 1030_COLUMNS.CSV
# COMPLETE JUPYTER NOTEBOOK
# ============================================================


# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import warnings
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


# ============================================================
# CELL 2 — DOWNLOAD NLTK RESOURCES
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources downloaded.")


# ============================================================
# CELL 3 — LOAD DATASET
# ============================================================

FILE_PATH = "1030_columns.csv"

df = pd.read_csv(FILE_PATH)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Number of rows    :", df.shape[0])
print("Number of columns :", df.shape[1])

display(df.head())


# ============================================================
# CELL 4 — BASIC DATASET INFORMATION
# ============================================================

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nShape:")
print(df.shape)

print("\nData types:")
display(df.dtypes.value_counts())

print("\nMissing values:")
missing_values = df.isnull().sum().sort_values(ascending=False)

display(
    missing_values.head(30).to_frame("missing_values")
)

print("\nFirst 30 column names:")

for i, column in enumerate(df.columns[:30], start=1):
    print(i, ":", column)


# ============================================================
# CELL 5 — IDENTIFY TEXT COLUMNS
# ============================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("=" * 70)
print("TEXT COLUMNS")
print("=" * 70)

print("Number of text columns:", len(text_columns))

for column in text_columns:
    print(column)


# ============================================================
# CELL 6 — IDENTIFY NUMERIC COLUMNS
# ============================================================

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

print("=" * 70)
print("NUMERIC COLUMNS")
print("=" * 70)

print("Number of numeric columns:", len(numeric_columns))

for column in numeric_columns[:50]:
    print(column)


# ============================================================
# CELL 7 — REMOVE EMPTY TEXT COLUMNS
# ============================================================

valid_text_columns = []

for column in text_columns:

    values = df[column].dropna().astype(str).str.strip()

    if len(values) > 0 and (values != "").any():
        valid_text_columns.append(column)

text_columns = valid_text_columns

print("=" * 70)
print("VALID TEXT COLUMNS")
print("=" * 70)

print("Number of usable text columns:", len(text_columns))


# ============================================================
# CELL 8 — NLP SETUP
# ============================================================

stop_words = set(
    stopwords.words("english")
)

lemmatizer = WordNetLemmatizer()


# ============================================================
# CELL 9 — NLP CLEANING FUNCTION
# ============================================================

def clean_text(text):

    if pd.isna(text):
        return ""

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove punctuation and numbers
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and very short words
    words = [
        word
        for word in words
        if word not in stop_words
        and len(word) > 2
    ]

    # Lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


# ============================================================
# CELL 10 — COMBINE ALL TEXT COLUMNS
# ============================================================

print("=" * 70)
print("COMBINING TEXT COLUMNS")
print("=" * 70)

if len(text_columns) == 0:

    raise ValueError(
        "No text columns were found in the dataset."
    )

df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

print("Text columns combined successfully.")

display(
    df[["combined_text"]].head()
)


# ============================================================
# CELL 11 — CLEAN TEXT
# ============================================================

print("=" * 70)
print("CLEANING TEXT")
print("=" * 70)

df["clean_text"] = (
    df["combined_text"]
    .apply(clean_text)
)

print("NLP preprocessing completed.")

display(
    df[
        ["combined_text", "clean_text"]
    ].head()
)


# ============================================================
# CELL 12 — CHECK TEXT LENGTH
# ============================================================

df["text_length"] = (
    df["clean_text"]
    .str.len()
)

df["word_count"] = (
    df["clean_text"]
    .str.split()
    .str.len()
)

print("=" * 70)
print("TEXT STATISTICS")
print("=" * 70)

print(
    "Records with no usable text:",
    (df["word_count"] == 0).sum()
)

print(
    "Average number of words:",
    round(df["word_count"].mean(), 2)
)

print(
    "Maximum number of words:",
    df["word_count"].max()
)


# ============================================================
# CELL 13 — REMOVE EMPTY TEXT RECORDS
# ============================================================

nlp_df = df[
    df["word_count"] > 0
].copy()

print("=" * 70)
print("NLP DATASET")
print("=" * 70)

print("Original rows :", len(df))
print("NLP rows      :", len(nlp_df))
print("Removed rows  :", len(df) - len(nlp_df))


# ============================================================
# CELL 14 — TF-IDF VECTORIZATION
# ============================================================

print("=" * 70)
print("TF-IDF VECTORIZATION")
print("=" * 70)

vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    nlp_df["clean_text"]
)

print("TF-IDF matrix shape:")
print(X.shape)

print(
    "Number of features:",
    len(vectorizer.get_feature_names_out())
)


# ============================================================
# CELL 15 — VIEW TF-IDF FEATURES
# ============================================================

feature_names = (
    vectorizer
    .get_feature_names_out()
)

print("=" * 70)
print("TF-IDF FEATURES")
print("=" * 70)

print(
    "First 100 features:"
)

print(
    feature_names[:100]
)


# ============================================================
# CELL 16 — DETERMINE K USING ELBOW METHOD
# ============================================================

print("=" * 70)
print("ELBOW METHOD")
print("=" * 70)

# Number of clusters to test
k_values = range(2, 11)

inertias = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertias.append(
        model.inertia_
    )

    print(
        f"k = {k} | Inertia = {model.inertia_:.2f}"
    )


# Plot elbow curve

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (k)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for K-Means"
)

plt.xticks(
    list(k_values)
)

plt.grid(
    True
)

plt.show()


# ============================================================
# CELL 17 — SILHOUETTE SCORE
# ============================================================

print("=" * 70)
print("SILHOUETTE SCORE")
print("=" * 70)

silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(score)

    print(
        f"k = {k} | Silhouette Score = {score:.4f}"
    )


# Plot silhouette scores

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (k)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(
    list(k_values)
)

plt.grid(
    True
)

plt.show()


# ============================================================
# CELL 18 — SELECT BEST K
# ============================================================

best_k = list(k_values)[
    np.argmax(silhouette_scores)
]

best_score = max(
    silhouette_scores
)

print("=" * 70)
print("BEST K")
print("=" * 70)

print(
    "Best number of clusters:",
    best_k
)

print(
    "Best silhouette score:",
    round(best_score, 4)
)


# ============================================================
# CELL 19 — RUN FINAL K-MEANS
# ============================================================

print("=" * 70)
print("FINAL K-MEANS MODEL")
print("=" * 70)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(
    X
)

nlp_df["cluster"] = cluster_labels

print("K-Means clustering completed.")


# ============================================================
# CELL 20 — CLUSTER COUNTS
# ============================================================

print("=" * 70)
print("CLUSTER DISTRIBUTION")
print("=" * 70)

cluster_counts = (
    nlp_df["cluster"]
    .value_counts()
    .sort_index()
)

display(
    cluster_counts.to_frame(
        "number_of_records"
    )
)


# ============================================================
# CELL 21 — CLUSTER PERCENTAGES
# ============================================================

cluster_percentages = (
    nlp_df["cluster"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

cluster_distribution = pd.DataFrame({
    "number_of_records": cluster_counts,
    "percentage": cluster_percentages.round(2)
})

display(
    cluster_distribution
)


# ============================================================
# CELL 22 — VISUALIZE CLUSTER DISTRIBUTION
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Records"
)

plt.title(
    "Number of Records in Each Cluster"
)

plt.grid(
    axis="y"
)

plt.show()


# ============================================================
# CELL 23 — FIND IMPORTANT TERMS IN EACH CLUSTER
# ============================================================

print("=" * 70)
print("TOP TERMS FOR EACH CLUSTER")
print("=" * 70)

terms = (
    vectorizer
    .get_feature_names_out()
)

order_centroids = (
    kmeans
    .cluster_centers_
    .argsort()[:, ::-1]
)

TOP_TERMS = 20

cluster_keywords = {}

for cluster_id in range(best_k):

    top_terms = [
        terms[index]
        for index in
        order_centroids[
            cluster_id,
            :TOP_TERMS
        ]
    ]

    cluster_keywords[
        cluster_id
    ] = top_terms

    print()
    print(
        "=" * 70
    )

    print(
        f"CLUSTER {cluster_id}"
    )

    print(
        "=" * 70
    )

    print(
        ", ".join(top_terms)
    )


# ============================================================
# CELL 24 — CREATE CLUSTER SUMMARY
# ============================================================

summary = []

for cluster_id in range(best_k):

    number_of_records = (
        nlp_df["cluster"]
        .eq(cluster_id)
        .sum()
    )

    keywords = cluster_keywords[
        cluster_id
    ]

    summary.append({
        "cluster": cluster_id,
        "number_of_records": number_of_records,
        "percentage": round(
            number_of_records /
            len(nlp_df) * 100,
            2
        ),
        "top_keywords": ", ".join(
            keywords
        )
    })

cluster_summary = pd.DataFrame(
    summary
)

display(
    cluster_summary
)


# ============================================================
# CELL 25 — SHOW SAMPLE RECORDS FROM EACH CLUSTER
# ============================================================

print("=" * 70)
print("SAMPLE RECORDS FROM EACH CLUSTER")
print("=" * 70)

for cluster_id in range(best_k):

    print()
    print(
        "=" * 70
    )

    print(
        f"CLUSTER {cluster_id}"
    )

    print(
        "=" * 70
    )

    sample = (
        nlp_df[
            nlp_df["cluster"]
            == cluster_id
        ]
        [["clean_text", "cluster"]]
        .head(5)
    )

    display(
        sample
    )


# ============================================================
# CELL 26 — CREATE 2D VISUALIZATION USING SVD
# ============================================================

print("=" * 70)
print("CREATING 2D CLUSTER VISUALIZATION")
print("=" * 70)

# Reduce TF-IDF dimensions to 2
svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

print(
    "Explained variance ratio:",
    svd.explained_variance_ratio_
)


# ============================================================
# CELL 27 — PLOT K-MEANS CLUSTERS
# ============================================================

plt.figure(
    figsize=(12, 8)
)

for cluster_id in range(best_k):

    mask = (
        nlp_df["cluster"]
        == cluster_id
    )

    plt.scatter(
        X_2d[mask, 0],
        X_2d[mask, 1],
        label=f"Cluster {cluster_id}",
        alpha=0.6
    )

plt.xlabel(
    "SVD Component 1"
)

plt.ylabel(
    "SVD Component 2"
)

plt.title(
    "K-Means Clusters — TF-IDF + SVD"
)

plt.legend()

plt.grid(
    True
)

plt.show()


# ============================================================
# CELL 28 — ADD CLUSTERS TO ORIGINAL DATASET
# ============================================================

print("=" * 70)
print("ADDING CLUSTERS TO ORIGINAL DATASET")
print("=" * 70)

# Create empty cluster column
df["cluster"] = pd.NA

# Match cluster labels using original index
df.loc[
    nlp_df.index,
    "cluster"
] = nlp_df["cluster"]

# Convert to nullable integer
df["cluster"] = (
    df["cluster"]
    .astype("Int64")
)

print(
    "Cluster labels added successfully."
)

display(
    df.head()
)


# ============================================================
# CELL 29 — CREATE HUMAN-READABLE CLUSTER NAMES
# ============================================================

# Automatically create names based on top 3 keywords

cluster_names = {}

for cluster_id in range(best_k):

    keywords = cluster_keywords[
        cluster_id
    ]

    name = (
        f"Cluster {cluster_id}: "
        + ", ".join(
            keywords[:3]
        )
    )

    cluster_names[
        cluster_id
    ] = name

print("=" * 70)
print("CLUSTER NAMES")
print("=" * 70)

for cluster_id, name in cluster_names.items():

    print(
        cluster_id,
        "->",
        name
    )


# ============================================================
# CELL 30 — ADD CLUSTER NAME
# ============================================================

df["cluster_name"] = (
    df["cluster"]
    .map(cluster_names)
)

display(
    df[
        [
            "cluster",
            "cluster_name"
        ]
    ].head(20)
)


# ============================================================
# CELL 31 — SAVE FULL CLUSTERED DATASET
# ============================================================

OUTPUT_FILE = (
    "1030_columns_kmeans_clusters.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 70)
print("DATASET SAVED")
print("=" * 70)

print(
    OUTPUT_FILE
)


# ============================================================
# CELL 32 — SAVE CLUSTER SUMMARY
# ============================================================

SUMMARY_FILE = (
    "1030_columns_cluster_summary.csv"
)

cluster_summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    "Cluster summary saved to:",
    SUMMARY_FILE
)


# ============================================================
# CELL 33 — SAVE CLUSTER KEYWORDS
# ============================================================

keyword_rows = []

for cluster_id in range(best_k):

    for rank, keyword in enumerate(
        cluster_keywords[cluster_id],
        start=1
    ):

        keyword_rows.append({
            "cluster": cluster_id,
            "rank": rank,
            "keyword": keyword
        })

cluster_keywords_df = pd.DataFrame(
    keyword_rows
)

KEYWORDS_FILE = (
    "1030_columns_cluster_keywords.csv"
)

cluster_keywords_df.to_csv(
    KEYWORDS_FILE,
    index=False
)

print(
    "Cluster keywords saved to:",
    KEYWORDS_FILE
)


# ============================================================
# CELL 34 — FINAL RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)

print(
    f"Original dataset: {df.shape[0]} rows × {df.shape[1]} columns"
)

print(
    f"Text columns used: {len(text_columns)}"
)

print(
    f"Records clustered: {len(nlp_df)}"
)

print(
    f"Number of clusters: {best_k}"
)

print(
    f"Silhouette score: {best_score:.4f}"
)

print("\nCluster distribution:")
display(
    cluster_distribution
)

print("\nCluster keywords:")
display(
    cluster_summary
)

print("\nOutput files:")
print(
    "1. 1030_columns_kmeans_clusters.csv"
)
print(
    "2. 1030_columns_cluster_summary.csv"
)
print(
    "3. 1030_columns_cluster_keywords.csv"
)


# ============================================================
# CELL 35 — OPTIONAL: LOAD RESULTS AGAIN
# ============================================================

# You can use this cell later to reload the final results.

final_df = pd.read_csv(
    "1030_columns_kmeans_clusters.csv"
)

print(
    "Final clustered dataset loaded:"
)

print(
    final_df.shape
)

display(
    final_df.head()
)

Libraries imported successfully.
NLTK resources downloaded.
DATASET LOADED
Number of rows    : 9
Number of columns : 1030


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,header 0,header 1,header 2,header 3,header 4,header 5,header 6,header 7,header 8,header 9,...,header 1020,header 1021,header 1022,header 1023,header 1024,header 1025,header 1026,header 1027,header 1028,header 1029
0,row 1 col 0,row 1 col 1,row 1 col 2,row 1 col 3,row 1 col 4,row 1 col 5,row 1 col 6,row 1 col 7,row 1 col 8,row 1 col 9,...,row 1 col 1020,row 1 col 1021,row 1 col 1022,row 1 col 1023,row 1 col 1024,row 1 col 1025,row 1 col 1026,row 1 col 1027,row 1 col 1028,row 1 col 1029
1,row 2 col 0,row 2 col 1,row 2 col 2,row 2 col 3,row 2 col 4,row 2 col 5,row 2 col 6,row 2 col 7,row 2 col 8,row 2 col 9,...,row 2 col 1020,row 2 col 1021,row 2 col 1022,row 2 col 1023,row 2 col 1024,row 2 col 1025,row 2 col 1026,row 2 col 1027,row 2 col 1028,row 2 col 1029
2,row 3 col 0,row 3 col 1,row 3 col 2,row 3 col 3,row 3 col 4,row 3 col 5,row 3 col 6,row 3 col 7,row 3 col 8,row 3 col 9,...,row 3 col 1020,row 3 col 1021,row 3 col 1022,row 3 col 1023,row 3 col 1024,row 3 col 1025,row 3 col 1026,row 3 col 1027,row 3 col 1028,row 3 col 1029
3,row 4 col 0,row 4 col 1,row 4 col 2,row 4 col 3,row 4 col 4,row 4 col 5,row 4 col 6,row 4 col 7,row 4 col 8,row 4 col 9,...,row 4 col 1020,row 4 col 1021,row 4 col 1022,row 4 col 1023,row 4 col 1024,row 4 col 1025,row 4 col 1026,row 4 col 1027,row 4 col 1028,row 4 col 1029
4,row 5 col 0,row 5 col 1,row 5 col 2,row 5 col 3,row 5 col 4,row 5 col 5,row 5 col 6,row 5 col 7,row 5 col 8,row 5 col 9,...,row 5 col 1020,row 5 col 1021,row 5 col 1022,row 5 col 1023,row 5 col 1024,row 5 col 1025,row 5 col 1026,row 5 col 1027,row 5 col 1028,row 5 col 1029


DATASET INFORMATION

Shape:
(9, 1030)

Data types:


str    1030
Name: count, dtype: int64

...

,combined_text
0,row 1 col 0 row 1 col 1 row 1 col 2 row 1 col ...
1,row 2 col 0 row 2 col 1 row 2 col 2 row 2 col ...
2,row 3 col 0 row 3 col 1 row 3 col 2 row 3 col ...
3,row 4 col 0 row 4 col 1 row 4 col 2 row 4 col ...
4,row 5 col 0 row 5 col 1 row 5 col 2 row 5 col ...


CLEANING TEXT
NLP preprocessing completed.


,combined_text,clean_text
0,row 1 col 0 row 1 col 1 row 1 col 2 row 1 col ...,row col row col row col row col row col row co...
1,row 2 col 0 row 2 col 1 row 2 col 2 row 2 col ...,row col row col row col row col row col row co...
2,row 3 col 0 row 3 col 1 row 3 col 2 row 3 col ...,row col row col row col row col row col row co...
3,row 4 col 0 row 4 col 1 row 4 col 2 row 4 col ...,row col row col row col row col row col row co...
4,row 5 col 0 row 5 col 1 row 5 col 2 row 5 col ...,row col row col row col row col row col row co...


TEXT STATISTICS
Records with no usable text: 0
Average number of words: 2060.0
Maximum number of words: 2060
NLP DATASET
Original rows : 9
NLP rows      : 9
Removed rows  : 0
TF-IDF VECTORIZATION


ValueError: After pruning, no terms remain. Try a lower min_df or a higher max_df.